# ISTAT SDMX API – Catalogo Dataflow

Questo notebook recupera tutti i **dataflow disponibili** dall'API SDMX di ISTAT e li carica in un DataFrame pandas per esplorazione e analisi.

**Riferimento:** [Guida API ISTAT – onData](https://ondata.github.io/guida-api-istat/)

**Endpoint base:** `https://esploradati.istat.it/SDMXWS/rest/dataflow/IT1`

> ⚠️ **Attenzione:** L'API ISTAT ha un limite di **5 richieste al minuto** per IP. Superarlo comporta un blocco di 1-2 giorni.

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time

print('Librerie caricate.')

In [ ]:
# -------------------------------------------------------------------
# Recupero XML dei dataflow dall'API ISTAT
# – stream=True per non aspettare l'intero body prima di iniziare
# – retry automatico con backoff esponenziale (max 4 tentativi)
# -------------------------------------------------------------------
import time

BASE_URL = 'https://esploradati.istat.it/SDMXWS/rest'
DATAFLOW_URL = f'{BASE_URL}/dataflow/IT1'

headers = {
    'Accept': 'application/xml',
    'User-Agent': 'Mozilla/5.0 (compatible; ISTAT-dataflow-script/1.0)'
}

MAX_RETRIES = 4
TIMEOUT     = 300   # secondi – la risposta XML può essere grande

raw_content = None

for attempt in range(1, MAX_RETRIES + 1):
    try:
        print(f'Tentativo {attempt}/{MAX_RETRIES} → {DATAFLOW_URL}')
        with requests.get(DATAFLOW_URL, headers=headers, timeout=TIMEOUT, stream=True) as r:
            r.raise_for_status()
            chunks = []
            total  = 0
            for chunk in r.iter_content(chunk_size=65536):
                if chunk:
                    chunks.append(chunk)
                    total += len(chunk)
                    print(f'  ricevuti {total / 1024:.0f} KB...', end='\r')
            raw_content = b''.join(chunks)
        print(f'\nRisposta completa – {len(raw_content) / 1024:.1f} KB  (status {r.status_code})')
        break  # successo

    except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
        wait = 2 ** attempt  # 2, 4, 8, 16 s
        print(f'\nErrore al tentativo {attempt}: {e}')
        if attempt < MAX_RETRIES:
            print(f'Riprovo tra {wait}s...')
            time.sleep(wait)
        else:
            raise RuntimeError('Tutti i tentativi falliti. Verifica la connessione o riprova più tardi.') from e

if raw_content is None:
    raise RuntimeError('Nessun contenuto ricevuto.')

In [ ]:
# -------------------------------------------------------------------
# Parsing XML SDMX 2.1
# -------------------------------------------------------------------
NS = {
    'mes':       'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message',
    'structure': 'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure',
    'common':    'http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common',
}

root = ET.fromstring(raw_content)  # usa raw_content dallo streaming

records = []

for df in root.findall('.//structure:Dataflow', NS):
    id_dataflow = df.get('id', '')
    version     = df.get('version', '')

    nome_it = ''
    nome_en = ''
    for name_el in df.findall('common:Name', NS):
        lang = name_el.get('{http://www.w3.org/XML/1998/namespace}lang', '')
        text = (name_el.text or '').strip()
        if lang == 'it':
            nome_it = text
        elif lang == 'en':
            nome_en = text

    id_datastructure = ''
    ref = df.find('.//structure:Structure/Ref', NS)
    if ref is None:
        ref = df.find('.//{http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure}Structure/Ref')
    if ref is None:
        ref = df.find('.//Ref')
    if ref is not None:
        id_datastructure = ref.get('id', '')

    url_istat = f'http://dati.istat.it/Index.aspx?DataSetCode={id_datastructure}' if id_datastructure else ''

    records.append({
        'iddataflow':      id_dataflow,
        'versione':        version,
        'descrizione':     nome_it,
        'descrizione_en':  nome_en,
        'iddatastructure': id_datastructure,
        'url':             url_istat,
    })

print(f'Dataflow trovati: {len(records)}')

In [ ]:
# -------------------------------------------------------------------
# Costruzione del DataFrame
# -------------------------------------------------------------------
df_dataflows = pd.DataFrame(records)

# Ordina per descrizione italiana
df_dataflows = df_dataflows.sort_values('descrizione').reset_index(drop=True)

print(f'DataFrame creato: {df_dataflows.shape[0]} righe × {df_dataflows.shape[1]} colonne')
print(f'\nColonne: {list(df_dataflows.columns)}')

df_dataflows.head(10)

In [ ]:
# -------------------------------------------------------------------
# Esplorazione del DataFrame
# -------------------------------------------------------------------

# Ricerca per parola chiave (modifica il termine qui sotto)
KEYWORD = 'popolazione'

mask = (
    df_dataflows['descrizione'].str.contains(KEYWORD, case=False, na=False) |
    df_dataflows['descrizione_en'].str.contains(KEYWORD, case=False, na=False)
)
risultati = df_dataflows[mask]

print(f'Risultati per "{KEYWORD}": {len(risultati)}')
risultati[['iddataflow', 'descrizione', 'descrizione_en', 'url']]

In [ ]:
# -------------------------------------------------------------------
# (Opzionale) Salva il catalogo in CSV
# -------------------------------------------------------------------
OUTPUT_FILE = 'istat_dataflows.csv'
df_dataflows.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Catalogo salvato in: {OUTPUT_FILE}')